# D1：教师原生轨迹生成（SFT 数据）

让教师模型（OpenAI 兼容 API，默认 DeepSeek-V4-Flash）在**真实 `swegemma` harness** 中跑 **Train 池（63 题，不含 dev60 / Val）**，产出可直接用于 SFT 的轨迹。

**输入**
- 比赛数据 `tasks.jsonl` + `snapshots/`，只使用 Train 池题目，dev60 与 Val 由代码硬性排除
- 挂载的 submission：`agent.yaml`、`prompts/system.md`、`eval_config.yaml`。工具、System Prompt 和预算都与线上保持一致
- 教师 API：在 Config cell 中填写 `BASE_URL / MODEL_NAME`；Key 放在 Kaggle Secrets，名称为 `TEACHER_API_KEY`

**输出**（`/kaggle/working/d1/`）
- `d1_teacher_sft.jsonl`：**验证通过**的轨迹，每行一条，可直接用于 SFT。字段为 `messages`（OpenAI chat 格式）、`tools`、`quality.qualified` 等
- `d1_attempts.jsonl`：每次 attempt 的台账（状态、原因、指标、费用），用于断点续跑
- `d1_split.json`：train / val 划分；`runs/<sample_id>/`：原始请求日志、harness trace 和 patch，供排查和 D3 复用
- `d1_results.zip`：以上全部内容打包

**运行前**
1. 挂载以下数据：比赛数据、`metric/gemma-4-developer-agent-wheelhouse`、`tianyuxu2/gemma-4-extra-test-wheels`、你的 submission 数据集（应为最终版：6 个工具、22 calls）。加速器选 **None (CPU)**，**Internet 打开**
2. 先跑冒烟：在 Config cell 中设 `TASK_LIMIT = 10, MAX_ATTEMPTS_PER_TASK = 2`，核对 token 用量和费用
3. 主采样：恢复默认值后 *Save & Run All*

**断点续跑**：`success` 和 `rejected`（教师自己没做成）不会重跑；`api_error` 和 `crash` 以及尚未运行的 seed 会在下次运行时自动补跑。如果是在新的 Kaggle 会话中继续，需要把上一个版本的 Output 挂载为输入，`RESUME_FROM='auto'` 会自动合并之前的结果。

## 1. Imports & Setup
安装 harness wheelhouse（离线 wheel），并修复测试环境，保证 Phase 2 的 `resolved` 可信。与 dev-eval notebook 的 cell 1 和 cell 2c 相同。

In [ ]:
# ==== 1. Imports & Setup：安装 harness wheelhouse，修复测试环境（与 dev-eval notebook 相同） ====
import asyncio
import concurrent.futures
import copy
import glob
import http.server
import importlib
import json
import logging
import os
import random
import re
import shutil
import site
import subprocess
import sys
import tempfile
import threading
import time
import venv
import zipfile
from collections import Counter
from pathlib import Path

NOTEBOOK_START = time.time()

os.environ.update({
    'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1', 'TRANSFORMERS_OFFLINE': '1',
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'LITELLM_TELEMETRY': 'False', 'LITELLM_LOG': 'WARNING',
    'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true',
})
# harness 内部的 LiteLLM 只会访问本地日志代理，真实 key 由代理注入，这里统一填占位值
os.environ['OPENAI_API_KEY'] = os.environ['LOCAL_API_KEY'] = 'EMPTY'


def configure_logging() -> None:
    for name in ('LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm', 'httpx', 'httpcore',
                 'openai', 'urllib3', 'opentelemetry', 'opentelemetry.context'):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate, lg.handlers[:] = False, []


configure_logging()

# --- harness wheelhouse（swegemma / adk-submission / google-adk / litellm） ---
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pth in glob.glob('/usr/local/lib/python*/*-packages/*cutlass*.pth'):
    Path(pth).unlink(missing_ok=True)
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    # Kaggle 上传数据集时会去掉 wheel 文件名中的 '+cu128'，这里恢复
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
configure_logging()

# --- 测试环境修复：补装 Kaggle 镜像缺失的测试依赖，并让 src-layout 仓库导入沙箱内的源码 ---
# 不修复时约 30% 的题会因环境问题判失败，Phase 2 的 resolved 信号就不可信
_extra = next(Path('/kaggle/input').rglob('inline_snapshot-*.whl'), None)
assert _extra is not None, '请挂载 tianyuxu2/gemma-4-extra-test-wheels 数据集'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(_extra.parent),
                'inline-snapshot', 'dirty-equals', 'pytest-httpbin', 'trustme'], check=True)
SRC_LAYOUT_HOOK = (
    "import os, sys; _s = [os.path.join(p, 'src') for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) "
    "if 'swegemma_sandbox_' in p and os.path.isdir(os.path.join(p, 'src'))]; "
    "sys.path[:0] = [p for p in _s if p not in sys.path]\n"
)
_sp = next(sp for sp in site.getsitepackages() if Path(sp).is_dir())
(Path(_sp) / '00_swegemma_src_layout.pth').write_text(SRC_LAYOUT_HOOK)

# 自检：在与 harness 相同的 system-site venv 中，依赖可以导入，且 src/ 排在 sys.path 最前面
_chk = Path(tempfile.mkdtemp(prefix='swegemma_sandbox_selfcheck_'))
(_chk / 'workspace' / 'src').mkdir(parents=True)
venv.create(_chk / 'venv', system_site_packages=True, with_pip=False, symlinks=True)
_res = subprocess.run([str(_chk / 'venv' / 'bin' / 'python3'), '-c',
                       'import sys, inline_snapshot, dirty_equals, pytest_httpbin, trustme; print(sys.path)'],
                      env={**os.environ, 'PYTHONPATH': str(_chk / 'workspace')}, capture_output=True, text=True)
assert _res.returncode == 0 and str(_chk / 'workspace' / 'src') in _res.stdout, f'sandbox self-check FAILED\n{_res.stderr[-2000:]}'
print('Setup OK: wheelhouse installed, test env fixed, sandbox self-check passed.')

## 2. Configuration
**本 notebook 唯一需要修改的 cell。** 包括教师 API、采样次数、并发、重试、熔断，以及 Train / Val 划分常量。

In [ ]:
# ==== 2. Configuration：本 notebook 唯一需要修改的 cell ====


def _secret(name: str) -> str:
    """优先读取 Kaggle Secrets，读不到时回退到同名环境变量。"""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, '')


# ---- 输入 / 输出 ----
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
SUBMISSION_INPUT = '/kaggle/input/datasets/youchenyyy/submission-1111'      # submission.zip 或包含 agent.yaml 的目录；None 表示在挂载数据中自动查找
OUT_DIR = Path('/kaggle/working/d1')
RESUME_FROM = 'auto'         # 'auto'：自动合并 /kaggle/input 下以前版本的 d1 输出；也可传路径列表；[] 表示不合并

# ---- 教师模型 API（OpenAI 兼容 /chat/completions） ----
BASE_URL = 'https://openrouter.ai/api/v1'
MODEL_NAME = 'deepseek/deepseek-v4-flash-0731'   # 按服务商实际的模型名填写；中段补采可换成 Pro 再跑一遍
API_KEY = _secret('TEACHER_API_KEY')
TEMPERATURE = 0.7
MAX_TOKENS = 2048                  # 单步最大生成长度（sft_plan §1.1）
EXTRA_BODY = {'reasoning': {'enabled': False}}
                   # 需要额外传给服务商的参数，例如关闭思考模式的开关
# harness 为 vLLM 准备的私有字段，外部 API 不认识，转发前删除
DROP_KEYS = ('chat_template_kwargs', 'thinking_token_budget', 'skip_special_tokens', 'include_reasoning',
             'reasoning_effort', 'top_k', 'min_p', 'repetition_penalty', 'max_completion_tokens')
PRICE_IN, PRICE_OUT = 0.14, 0.28   # 美元 / 百万 token，只用于费用估算

# ---- 采样（sft_plan §2.4） ----
BASE_SEED = 0
MAX_ATTEMPTS_PER_TASK = 8          # 每题最多尝试次数；第 k 次 attempt 使用 seed = BASE_SEED + k
TARGET_PER_TASK = 3                # 某题攒够这么多条合格轨迹后不再继续采样
TASK_LIMIT = None                  # 冒烟时设为 10（同时把 MAX_ATTEMPTS_PER_TASK 设为 2）
MAX_WORKERS = 3                    # 并发题数；CPU notebook 共 4 核，留 1 核给 harness
TEACHER_MAX_TIME_MINUTES = 10.0    # 教师 API 较慢，单题时间放宽；存档时 prompt 中的时间会改回线上值
PORT_BASE = 18500                  # 日志代理端口，每个 worker 占用一个

# ---- 重试（指数退避） ----
MAX_RETRIES = 6
BACKOFF_BASE, BACKOFF_MAX = 2.0, 60.0
REQUEST_TIMEOUT = 180

# ---- 熔断 ----
MAX_COST_USD = 15.0                # 本次运行的 API 费用达到上限后，不再发起新的 attempt
MAX_WALL_HOURS = 11.0              # Kaggle 会话 12 小时后会被强制结束

# 教师专用的 System Prompt 后缀（sft_plan §2.3），只发给教师，存档时不包含
TEACHER_SUFFIX = ('Before each tool call write at most 2 short sentences of reasoning. Use git grep -n to locate '
                  'symbols quickly. Edit a source file by call 8. Verify your fix with a targeted pytest (-k) '
                  'or /tmp/repro.py before submitting. Finish within 16 tool calls.')

# 合格轨迹的阈值（sft_plan §2.6）
QUALITY = dict(max_tool_calls=16, max_first_edit=9, max_total_tokens=14000, max_thought_chars=1000)
PLAN_TOOLS = {'run_command', 'read_file', 'edit_file', 'write_file', 'get_status', 'submit_patch'}
PLAN_MAX_TOOL_CALLS = 22

# ---- 数据划分：dev60 只用于评测，永远不进入训练 ----
DEV_IDS = [
    'fastapi_13713', 'rich_3063', 'requests_6629', 'httpx_3672', 'fastapi_11194', 'rich_3043',
    'fastapi_12942', 'rich_3067', 'fastapi_14246', 'fastapi_14262', 'rich_3105', 'fastapi_14349',
    'requests_6589', 'rich_2725', 'fastapi_5624', 'fastapi_14361', 'rich_3278', 'fastapi_14303',
    'rich_3468', 'fastapi_14297', 'rich_3470', 'fastapi_13207', 'requests_7205', 'fastapi_14430',
    'rich_3454', 'fastapi_14448', 'rich_3480', 'fastapi_14455', 'fastapi_14458', 'rich_3506',
    'fastapi_14482', 'rich_3535', 'requests_7328', 'fastapi_14487', 'rich_3675', 'fastapi_14371',
    'fastapi_14605', 'rich_3777', 'fastapi_14479', 'rich_3718', 'fastapi_14786', 'requests_7427',
    'fastapi_14794', 'rich_3934', 'fastapi_14616', 'rich_3894', 'fastapi_14873', 'rich_3938',
    'fastapi_14953', 'fastapi_14978', 'rich_3944', 'requests_7502', 'fastapi_15023', 'rich_4006',
    'fastapi_15280', 'fastapi_15589', 'rich_4075', 'fastapi_15745', 'rich_4077', 'fastapi_15785',
]
# 参考 patch 在本地环境中也无法通过的题（eval/env_broken.json），只能做弱验证
ENV_BROKEN = {
    'fastapi_14186', 'fastapi_14356', 'fastapi_15745', 'fastapi_15763', 'fastapi_15785', 'requests_6589',
    'requests_6592', 'requests_6629', 'requests_6757', 'rich_3064', 'rich_3130', 'rich_3296', 'rich_3469',
    'rich_3486', 'rich_3782',
}
VAL_PER_REPO = {'fastapi/fastapi': 2, 'Textualize/rich': 2, 'psf/requests': 1}   # 从严格可验题中抽取
SPLIT_SEED = 0

assert API_KEY, '未找到教师 API Key：请在 Kaggle Secrets 中添加 TEACHER_API_KEY'
assert len(set(DEV_IDS)) == 60

## 3. Data Loading & Checkpoint Filtering
- 构建 Train 池：129 题去掉 dev60 以及与 dev 共用 base_commit 的题，剩 68 题；再抽出 Val 5 题，剩下 **D1 = 63 题**
- 加载并校验 submission，工具、Prompt 和预算以它为准
- 读取已有输出：`success` 和 `rejected` 跳过，其余进入待跑队列

In [ ]:
# ==== 3. Data Loading & Checkpoint Filtering ====
import yaml
from adk_submission import discover_adapters, validate_directory
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, build_submission_limits
from swegemma.models import load_tasks

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
RUNS_DIR = OUT_DIR / 'runs'
SFT_PATH, LEDGER_PATH, SPLIT_PATH = OUT_DIR / 'd1_teacher_sft.jsonl', OUT_DIR / 'd1_attempts.jsonl', OUT_DIR / 'd1_split.json'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
FINAL = {'success', 'rejected'}   # 终态，不再重跑；api_error / crash 会在下次运行时重跑


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]


def append_jsonl(path: Path, rec: dict) -> None:
    """追加一行并立即落盘，即使会话中断也不会丢失已完成的结果。"""
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + '\n')
        f.flush()
        os.fsync(f.fileno())


def sample_id(iid: str, k: int) -> str:
    return f'{iid}__s{BASE_SEED + k}'


# ---- 3.1 Train / Val / D1 划分（sft_plan §2.1） ----
RAW = {d['instance_id']: d for d in read_jsonl(TASKS_PATH)}      # 原始字段，包括参考 patch，只用于弱验证
TASKS = {t.instance_id: t for t in load_tasks(TASKS_PATH)}        # harness 使用的 Task 对象
dev_commits = {RAW[i]['base_commit'] for i in DEV_IDS}
pool = sorted(i for i, d in RAW.items() if i not in DEV_IDS and d['base_commit'] not in dev_commits)
rng = random.Random(SPLIT_SEED)
VAL_IDS = []
for repo, n in VAL_PER_REPO.items():
    VAL_IDS += sorted(rng.sample([i for i in pool if RAW[i]['repo'] == repo and i not in ENV_BROKEN], n))
D1_IDS = [i for i in pool if i not in VAL_IDS]
rng.shuffle(D1_IDS)   # 打乱顺序，让 TASK_LIMIT 截取的前缀也覆盖多个仓库
assert not set(D1_IDS) & (set(DEV_IDS) | set(VAL_IDS)), 'D1 混入了 dev / val 题目'
SPLIT_PATH.write_text(json.dumps({'train_pool': pool, 'val': VAL_IDS, 'd1': sorted(D1_IDS), 'dev': DEV_IDS}, indent=1))
if TASK_LIMIT:
    D1_IDS = D1_IDS[:TASK_LIMIT]
print(f'Train pool {len(pool)} = D1 {len(pool) - len(VAL_IDS)} + Val {len(VAL_IDS)} {VAL_IDS}; this run: {len(D1_IDS)} tasks')
print('  D1 repos:', dict(Counter(RAW[i]['repo'] for i in D1_IDS)), '| weak-verify:', sorted(set(D1_IDS) & ENV_BROKEN))


# ---- 3.2 加载 submission：工具、Prompt 和预算都以它为准，保证与线上一致 ----
def _zip_has_agent(z: Path) -> bool:
    try:
        return 'agent.yaml' in zipfile.ZipFile(z).namelist()
    except zipfile.BadZipFile:
        return False


def find_submission() -> Path:
    """查找根目录含 agent.yaml 的 zip，或包含 agent.yaml 的目录（与 dev-eval notebook 的逻辑相同）。"""
    if SUBMISSION_INPUT:
        return Path(SUBMISSION_INPUT)
    inp = Path('/kaggle/input')
    roots = [p for p in (inp / 'datasets').glob('*/*') if p.is_dir()]
    roots += [p for p in inp.iterdir() if p.is_dir() and p.name not in ('competitions', 'models', 'datasets')]
    roots = [r for r in roots if r.resolve() != WHEELHOUSE_DIR.resolve()]
    zips = [z for r in roots for z in r.rglob('*.zip') if _zip_has_agent(z)]
    dirs = [p.parent for r in roots for p in r.rglob('agent.yaml')]
    cands = zips or dirs
    assert len(cands) == 1, f'应当恰好找到一个 submission，实际为 zips={zips} dirs={dirs}；请设置 SUBMISSION_INPUT'
    return cands[0]


SUBMISSION_SRC = find_submission()
AGENT_DIR = Path('/tmp/submission_src')   # 不放在 /kaggle/working，避免续跑时被当成第二个 submission
shutil.rmtree(AGENT_DIR, ignore_errors=True)
if SUBMISSION_SRC.suffix == '.zip':
    zipfile.ZipFile(SUBMISSION_SRC).extractall(AGENT_DIR)
else:
    shutil.copytree(SUBMISSION_SRC, AGENT_DIR)
limits, gen_constraints = build_submission_limits()
validate_directory(AGENT_DIR, limits=limits)
declared_model = next(iter(discover_declared_models(AGENT_DIR)))
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)


class _IncludeLoader(yaml.SafeLoader):
    """只用于读取工具列表：把 !include 当作普通字符串，不展开。"""


_IncludeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))
agent_cfg = yaml.load((AGENT_DIR / 'agent.yaml').read_text(encoding='utf-8'), Loader=_IncludeLoader)
_ec = AGENT_DIR / 'eval_config.yaml'
EVAL = (yaml.safe_load(_ec.read_text(encoding='utf-8')) or {}).get('evaluation', {}) if _ec.exists() else {}
TIMEOUT_SECONDS = int(EVAL.get('timeout_seconds', 300))
VERIFY_TIMEOUT_SECONDS = int(EVAL.get('verification_timeout_seconds', 300))
INFER_MAX_TIME_MINUTES = float(EVAL.get('max_time_minutes', 60.0))   # 线上时间预算，存档时写回 prompt
MAX_TOOL_CALLS = int(EVAL.get('max_tool_calls', 100))
MAX_TURNS = int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None

tools_declared = {t for t in agent_cfg.get('tools', []) if isinstance(t, str)}
print(f'Submission {SUBMISSION_SRC} | model={declared_model} | tools={sorted(tools_declared)}')
print(f'  budget: {MAX_TOOL_CALLS} calls, {INFER_MAX_TIME_MINUTES} min (teacher {TEACHER_MAX_TIME_MINUTES} min), '
      f'{MAX_TURNS} turns, cmd timeout {TIMEOUT_SECONDS}s')
if tools_declared != PLAN_TOOLS or MAX_TOOL_CALLS != PLAN_MAX_TOOL_CALLS:
    print(f'⚠️  submission 与 sft_plan 不一致（计划为 {sorted(PLAN_TOOLS)}、{PLAN_MAX_TOOL_CALLS} calls）。'
          '轨迹会严格按挂载的 submission 生成，请确认挂载的是最终版。')


# ---- 3.3 断点续跑：合并历史结果，只把未完成的 attempt 放入待跑队列 ----
def merge_ledger(rows: list[dict]) -> dict[str, dict]:
    """同一 sample_id 有多条记录时，终态优先；同为终态或同为非终态时，取后出现的一条。"""
    out = {}
    for r in rows:
        old = out.get(r['sample_id'])
        if old is None or r['status'] in FINAL or old['status'] not in FINAL:
            out[r['sample_id']] = r
    return out


prev_dirs = ([p.parent for p in Path('/kaggle/input').rglob('d1_attempts.jsonl')] if RESUME_FROM == 'auto'
             else [Path(p) for p in RESUME_FROM])
prev_dirs = [d for d in prev_dirs if d.resolve() != OUT_DIR.resolve()]
ledger = merge_ledger([r for d in [*prev_dirs, OUT_DIR] for r in read_jsonl(d / 'd1_attempts.jsonl')])
sft_rows = {r['sample_id']: r for d in [*prev_dirs, OUT_DIR] for r in read_jsonl(d / 'd1_teacher_sft.jsonl')}
for sid, r in sft_rows.items():   # 如果 SFT 已写入但台账没来得及写，以 SFT 文件为准
    if ledger.get(sid, {}).get('status') != 'success':
        ledger[sid] = {k: v for k, v in r.items() if k not in ('messages', 'tools', 'patch')}
for d in prev_dirs:               # 复制以前版本的原始日志（已存在的不覆盖）
    if (d / 'runs').is_dir():
        shutil.copytree(d / 'runs', RUNS_DIR, dirs_exist_ok=True)
# 去重后重写两个文件，保证后续追加写入的基础是干净的
for path, rows in ((LEDGER_PATH, ledger.values()), (SFT_PATH, sft_rows.values())):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(''.join(json.dumps(r, ensure_ascii=False, default=str) + '\n' for r in rows), encoding='utf-8')
    tmp.replace(path)


def n_qualified(iid: str) -> int:
    return sum(r['instance_id'] == iid and r['status'] == 'success' and r['quality']['qualified']
               for r in ledger.values())


def pending_attempts(iid: str) -> list[int]:
    """该题还需要跑的 attempt 编号（上限值；运行中攒够合格轨迹会提前停止）。"""
    if n_qualified(iid) >= TARGET_PER_TASK:
        return []
    return [k for k in range(MAX_ATTEMPTS_PER_TASK) if ledger.get(sample_id(iid, k), {}).get('status') not in FINAL]


_st = Counter(r['status'] for r in ledger.values())
print(f'Resumed from {len(prev_dirs)} previous output dir(s): {dict(_st)}')
print(f'Tasks at target: {sum(n_qualified(i) >= TARGET_PER_TASK for i in D1_IDS)}/{len(D1_IDS)} | '
      f'pending attempts (upper bound): {sum(len(pending_attempts(i)) for i in D1_IDS)}')

## 4. Teacher Agent Rollout Core
`harness (LiteLLM)` → **本地日志代理** → `教师 API`
- 代理：改写为线上视角（时间预算），只对教师追加 `TEACHER_SUFFIX` 并覆盖采样参数，429 / 5xx / 超时做指数退避重试，每个请求落盘到 `runs/<sample_id>/requests.jsonl`
- `run_attempt`：真实 harness 跑 Phase 1（智能体）和 Phase 2（验证），然后还原轨迹 → 判定（严格 / 弱验证）→ §2.6 质量指标
- 状态：`success` 写入 SFT 文件；`rejected` 表示教师失败或轨迹不合格，不重跑；`api_error` 和 `crash` 下次重跑

In [ ]:
# ==== 4. Teacher Agent Rollout Core ====
# 数据流：harness (LiteLLM) -> 本地日志代理 (127.0.0.1:PORT_BASE+w) -> 教师 API
# 代理负责：改写为线上视角、注入教师后缀和采样参数、指数退避重试、逐请求落盘。
# 每个 worker 独占一个端口，同一时刻只跑一个 attempt，所以按端口就能确定请求属于哪个 sample。
import httpx
import litellm
from adk_submission import VllmConfig, VllmServer
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator

litellm.drop_params = True
SLOTS: dict[int, dict] = {}   # port -> 当前 attempt 的上下文 {sample_id, seed, log, failed, retried}
TIME_RE = re.compile(r'(Time allowance: )[\d.]+( minutes)')
HTTP = httpx.Client(timeout=REQUEST_TIMEOUT)


def _map_text(content, fn):
    """对消息 content 中的文本应用 fn；content 可能是 str，也可能是 OpenAI 的 parts 列表。"""
    if isinstance(content, str):
        return fn(content)
    if isinstance(content, list):
        return [{**p, 'text': fn(p['text'])} if p.get('type') == 'text' else p for p in content]
    return content


def to_archive(messages: list) -> list:
    """线上视角的消息：把首条 user 中的时间预算改回线上值。教师看到的也是这一版。"""
    msgs = copy.deepcopy(messages)
    for m in msgs:
        if m.get('role') == 'user':
            m['content'] = _map_text(m.get('content'), lambda t: TIME_RE.sub(
                lambda x: f'{x.group(1)}{INFER_MAX_TIME_MINUTES}{x.group(2)}', t, count=1))
            break
    return msgs


def to_upstream(body: dict, archive: list, seed: int) -> dict:
    """实际发给教师的请求：在存档版基础上追加 TEACHER_SUFFIX，并覆盖采样参数。
    ADK 上下文压缩时发出的摘要请求没有 system 消息，这类请求不追加后缀，原样转发。"""
    msgs = copy.deepcopy(archive)
    sys_msg = next((m for m in msgs if m.get('role') in ('system', 'developer')), None)
    if sys_msg is not None:
        c = sys_msg.get('content')
        sys_msg['content'] = (c + '\n\n' + TEACHER_SUFFIX if isinstance(c, str)
                              else [*(c or []), {'type': 'text', 'text': TEACHER_SUFFIX}])
    send = {k: v for k, v in body.items() if k not in DROP_KEYS}
    send.update(model=MODEL_NAME, messages=msgs, temperature=TEMPERATURE, seed=seed, max_tokens=MAX_TOKENS, **EXTRA_BODY)
    return send


def upstream_url(path: str) -> str:
    return BASE_URL.rstrip('/') + (path[3:] if path.startswith('/v1/') else path)


def forward(method: str, url: str, payload: dict | None):
    """带指数退避和 jitter 的转发。429、5xx、超时和连接错误会重试。返回 (response 或 None, 重试次数, 错误信息)。"""
    headers = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}
    resp, err = None, ''
    for i in range(MAX_RETRIES + 1):
        try:
            resp = HTTP.request(method, url, headers=headers, json=payload)
            if resp.status_code != 429 and resp.status_code < 500:
                return resp, i, ''
            err = f'HTTP {resp.status_code}: {resp.text[:300]}'
        except httpx.HTTPError as e:
            resp, err = None, f'{type(e).__name__}: {e}'
        if i < MAX_RETRIES:
            time.sleep(min(BACKOFF_MAX, BACKOFF_BASE * 2 ** i) * random.uniform(0.8, 1.2))
    return resp, MAX_RETRIES, err


class LoggingProxy(http.server.BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'

    def log_message(self, *args):  # 不输出 access log
        pass

    def _reply(self, code: int, data: bytes) -> None:
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):  # 例如 /v1/models，原样透传
        resp, _, err = forward('GET', upstream_url(self.path), None)
        self._reply(resp.status_code if resp is not None else 502, resp.content if resp is not None else err.encode())

    def do_POST(self):
        slot = SLOTS.get(self.server.server_address[1])
        body = json.loads(self.rfile.read(int(self.headers.get('Content-Length') or 0)) or b'{}')
        if body.get('stream') or slot is None:
            msg = 'streaming is not supported' if body.get('stream') else 'no active sample on this port'
            return self._reply(400, json.dumps({'error': {'message': f'logging proxy: {msg}'}}).encode())
        try:
            code, data = self._proxy(slot, body)
        except Exception as e:   # 代理自身出错：记录原因并返回 500，而不是直接断开连接
            slot['failed'] += 1
            slot['proxy_error'] = f'{type(e).__name__}: {e}'[:300]
            code, data = 500, json.dumps({'error': {'message': f"logging proxy: {slot['proxy_error']}"}}).encode()
        self._reply(code, data)

    def _proxy(self, slot: dict, body: dict) -> tuple[int, bytes]:
        archive = to_archive(body.get('messages', []))
        t0 = time.time()
        resp, retries, err = forward('POST', upstream_url(self.path), to_upstream(body, archive, slot['seed']))
        try:
            data = resp.json() if resp is not None else None
        except ValueError:
            data = None
        ok = resp is not None and resp.status_code == 200 and isinstance(data, dict) and data.get('choices')
        slot['retried'] += retries > 0
        slot['failed'] += not ok
        append_jsonl(slot['log'], {
            'ts': t0, 'latency_s': round(time.time() - t0, 2), 'status': resp.status_code if resp is not None else None,
            'retries': retries, 'error': err if not ok else '', 'messages': archive, 'tools': body.get('tools'),
            'response': data if ok else None, 'usage': (data or {}).get('usage') if ok else None,
        })
        if resp is None:
            return 502, json.dumps({'error': {'message': err}}).encode()
        return resp.status_code, resp.content


def start_proxies() -> None:
    global PROXIES
    for srv in globals().get('PROXIES', []):   # 重复执行本 cell 时，先关闭旧的代理
        srv.shutdown()
        srv.server_close()
    PROXIES = [http.server.ThreadingHTTPServer(('127.0.0.1', PORT_BASE + w), LoggingProxy) for w in range(MAX_WORKERS)]
    for srv in PROXIES:
        threading.Thread(target=srv.serve_forever, daemon=True).start()


def make_registry(port: int):
    """用与 dev-eval 相同的 create_model_registry 构造模型注册表（不启动 vLLM），把 endpoint 指向本地代理。
    这样 LiteLlm 的参数与线上完全一致，只是请求发到了教师 API。"""
    try:
        srv = VllmServer(VllmConfig(model=declared_model, host='127.0.0.1', port=port), adapter_manifest=adapters)
        return srv.create_model_registry(aliases=[declared_model], model_prefix='openai/', api_key='EMPTY')
    except Exception as e:  # 退路：直接构造 LiteLlm
        print(f'create_model_registry failed ({type(e).__name__}: {e}); fallback to LiteLlm registry')
        from adk_submission import ModelRegistry
        from google.adk.models.lite_llm import LiteLlm
        reg = ModelRegistry()
        reg.register(declared_model, LiteLlm(model=f'openai/{declared_model}', api_base=f'http://127.0.0.1:{port}/v1',
                                             api_key='EMPTY', num_retries=3))
        return reg


# ---------------- 轨迹解析 ----------------
PROTECTED_RE = re.compile(r'(^|/)(tests?|testing)/.*\.py$|(^|/)test_[^/]*\.py$|_test\.py$|(^|/)conftest\.py$')
BROKEN_CODE_RE = re.compile(r'^E\s+(SyntaxError|IndentationError|TabError)\b', re.M)
API_ERR_RE = re.compile(r'RateLimitError|APIConnectionError|APITimeoutError|ServiceUnavailableError|'
                        r'InternalServerError|BadGatewayError|logging proxy', re.I)
BARE_PYTEST_RE = re.compile(r'(?:^|[;&|(]|\s)(?:python3? -m )?pytest(?:\s+-\S+)*\s*(?:$|[;&|)])|unittest discover', re.M)
VERIFY_RE = re.compile(r'pytest\s+\S+\.py|python3?\s+/tmp/\S+\.py')
SEARCH_RE = re.compile(r'\b(grep|rg|find|cat|sed -n|ls|head|tail)\b')
SKIP_COUNT = {'submit_patch', 'get_status'}   # 不计入工具调用预算


def tool_payload(msg: dict) -> dict:
    """解析 tool 消息中的 JSON。ADK 可能多包一层 {"result": "<json>"}，这里一并解开。"""
    try:
        d = json.loads(msg.get('content') or '{}')
        if isinstance(d, dict) and set(d) == {'result'} and isinstance(d['result'], str):
            d = json.loads(d['result'])
        return d if isinstance(d, dict) else {}
    except (TypeError, ValueError):
        return {}


def iter_calls(messages: list):
    """按顺序产出 (tool_call, 参数 dict 或 None, 对应 tool 消息的 payload)。"""
    results = {m.get('tool_call_id'): tool_payload(m) for m in messages if m.get('role') == 'tool'}
    for m in messages:
        for tc in (m.get('tool_calls') or []) if m.get('role') == 'assistant' else []:
            try:
                args = json.loads(tc['function'].get('arguments') or '{}')
            except ValueError:
                args = None
            yield tc, args, results.get(tc.get('id'), {})


def build_trajectory(reqs: list[dict]):
    """把请求日志还原成一条完整轨迹：取最后一个主请求的 messages，加上它的回复。
    返回 (messages, tools, 最后一次请求的 usage, 拒绝原因)。拒绝原因为 None 表示通过基础质量检查。"""
    main = [r for r in reqs if r.get('tools') and r.get('response')]
    if not main:
        return None, None, None, 'no_agent_requests'
    for a, b in zip(main, main[1:]):   # 前缀性质被破坏，说明发生了上下文压缩
        if b['messages'][:len(a['messages'])] != a['messages']:
            return None, None, None, 'compacted'
    choice = main[-1]['response']['choices'][0]
    reply = {k: v for k, v in choice['message'].items() if k in ('role', 'content', 'tool_calls', 'reasoning_content') and v}
    messages = main[-1]['messages'] + [{**reply, 'role': 'assistant'}]
    usage = main[-1].get('usage') or {}

    if any(r['response']['choices'][0].get('finish_reason') == 'length' for r in main):
        return None, None, None, 'truncated_output'
    if sum(m.get('role') == 'user' for m in messages) > 1:
        return None, None, None, 'nudged'   # 首条之后又出现 user 消息，说明 harness 发过 nudge（中途有空回复或输出被截断）
    if any(args is None for _, args, _ in iter_calls(messages)):
        return None, None, None, 'bad_tool_args'
    # 截断到第一次成功的 submit_patch 为止；如果紧随其后是纯文本收尾回复，则保留
    sub_ids = {tc['id'] for tc, _, res in iter_calls(messages)
               if tc['function']['name'] == 'submit_patch' and res.get('status') == 'ok' and res.get('patch_size', 0) > 0}
    end = next((i + 1 for i, m in enumerate(messages) if m.get('role') == 'tool' and m.get('tool_call_id') in sub_ids), None)
    if end is None:
        return None, None, None, 'no_submit'
    while end < len(messages) and messages[end].get('role') == 'tool':
        end += 1
    if end < len(messages) and messages[end].get('role') == 'assistant' and not messages[end].get('tool_calls'):
        end += 1
    messages = messages[:end]
    last = messages[-1]
    if last.get('role') != 'assistant' or last.get('tool_calls') or not str(last.get('content') or '').strip():
        return None, None, None, 'no_final_reply'
    return messages, main[-1]['tools'], usage, None


# ---------------- 判定与质量指标 ----------------
HUNK_RE = re.compile(r'^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@')


def edited_lines(diff: str) -> dict[str, set[int]]:
    """统一 diff -> {源码文件: 被修改的原文件行号}，忽略测试文件（精简自 eval/refdiff.py）。"""
    files, path, old_path, old_no, ol, nl = {}, None, None, 0, 0, 0
    for line in (diff or '').splitlines():
        if (ol > 0 or nl > 0) and path:   # 按 hunk 头中的行数消费内容，避免把 '---' 开头的内容行当成文件头
            tag = line[:1] or ' '
            if tag == '\\':
                continue
            if tag in '- ':
                if tag == '-':
                    files[path].add(old_no)
                old_no, ol, nl = old_no + 1, ol - 1, nl - (tag == ' ')
                continue
            if tag == '+':
                files[path].update((max(1, old_no - 1), old_no))
                nl -= 1
                continue
            ol = nl = 0   # 格式异常：结束当前 hunk，把这一行当作头部重新解析
        if line.startswith('--- '):
            old_path = line[4:].split('\t')[0].strip()
        elif line.startswith('+++ '):
            p = line[4:].split('\t')[0].strip()
            p = old_path if p == '/dev/null' else p
            path = p[2:] if p.startswith(('a/', 'b/')) else p
        elif (m := HUNK_RE.match(line)) and path:
            old_no, ol, nl = int(m.group(1)), int(m.group(2) or 1), int(m.group(4) or 1)
            files.setdefault(path, set())
    return {p: l for p, l in files.items() if not PROTECTED_RE.search(p)}


def verify(iid: str, patch: str, resolved: bool, test_log: str, error: str) -> dict:
    """严格可验题看 resolved；弱验证题（ENV_BROKEN）要求编辑位置与参考 patch 一致（same_lines），
    没有多余文件，且测试日志中没有语法错误。"""
    if iid not in ENV_BROKEN or resolved:
        return {'mode': 'strict', 'passed': bool(resolved)}
    ref, agt = edited_lines(RAW[iid]['patch']), edited_lines(patch)
    dist = min((abs(a - b) for p in ref.keys() & agt.keys() for a in agt[p] for b in ref[p]), default=10 ** 9)
    extra = sorted(set(agt) - set(ref))
    broken = bool(BROKEN_CODE_RE.search(test_log or ''))
    apply_fail = 'apply' in (error or '').lower()
    return {'mode': 'weak', 'passed': bool(patch) and dist <= 3 and not extra and not broken and not apply_fail,
            'min_line_dist': dist, 'extra_files': extra, 'syntax_error': broken, 'apply_fail': apply_fail}


def quality(messages: list, usage: dict) -> dict:
    """sft_plan §2.6 的轨迹指标；qualified 表示全部达标。"""
    step, first_edit, edit_fail, recovered, tampered, bare_pytest, verified = 0, None, 0, False, False, False, False
    seen, dup = set(), False
    for tc, args, res in iter_calls(messages):
        name = tc['function']['name']
        if name in SKIP_COUNT:
            continue
        step += 1
        path = str(args.get('filepath', '')).removeprefix('/workspace/').lstrip('/')
        cmd = str(args.get('command', ''))
        if name in ('edit_file', 'write_file'):
            if PROTECTED_RE.search(path):
                tampered = True
            elif res.get('status') == 'ok' and first_edit is None and not path.startswith('tmp/'):
                first_edit = step
            if name == 'edit_file':
                if res.get('status') != 'ok':
                    edit_fail += 1
                elif edit_fail:
                    recovered = True
        if name == 'run_command':
            bare_pytest |= bool(BARE_PYTEST_RE.search(cmd))
            verified |= bool(VERIFY_RE.search(cmd)) and first_edit is not None
        if name == 'read_file' or (name == 'run_command' and SEARCH_RE.search(cmd)):
            key = (name, json.dumps(args, sort_keys=True))
            dup |= key in seen
            seen.add(key)
    thought = max((len(str(m.get('content') or '')) for m in messages if m.get('role') == 'assistant' and m.get('tool_calls')), default=0)
    total_tokens = int(usage.get('total_tokens') or 0)
    q = {'tool_calls': step, 'first_edit_step': first_edit, 'edit_fail': edit_fail, 'total_tokens': total_tokens,
         'max_thought_chars': thought, 'has_targeted_verify': verified, 'duplicate_search': dup,
         'bare_pytest': bare_pytest, 'test_tampered': tampered}
    rules = {
        'tool_calls': step <= QUALITY['max_tool_calls'],
        'first_edit': first_edit is not None and first_edit <= QUALITY['max_first_edit'],
        'total_tokens': total_tokens <= QUALITY['max_total_tokens'],
        'thought_len': thought <= QUALITY['max_thought_chars'],
        'no_duplicate': not dup, 'no_bare_pytest': not bare_pytest, 'no_test_tamper': not tampered,
        'edit_recovery': edit_fail == 0 or (edit_fail == 1 and recovered),
    }
    return {**q, 'failed_rules': [k for k, v in rules.items() if not v], 'qualified': all(rules.values())}


def summarize_usage(reqs: list[dict]) -> dict:
    p = sum(int((r.get('usage') or {}).get('prompt_tokens') or 0) for r in reqs)
    c = sum(int((r.get('usage') or {}).get('completion_tokens') or 0) for r in reqs)
    return {'requests': len(reqs), 'prompt_tokens': p, 'completion_tokens': c,
            'cost_usd': round((p * PRICE_IN + c * PRICE_OUT) / 1e6, 5)}


def _maybe(obj, *names):
    """harness 结果对象的字段名可能因版本而不同，依次尝试，返回第一个非 None 的值。"""
    return next((v for n in names if (v := getattr(obj, n, None)) is not None), None)


# ---------------- 单次 attempt ----------------
async def run_attempt(iid: str, k: int, port: int) -> dict:
    """在真实 harness 中跑一次教师 rollout（Phase 1 智能体 + Phase 2 验证），返回 SFT 记录或失败记录。"""
    sid, seed = sample_id(iid, k), BASE_SEED + k
    run_dir = RUNS_DIR / sid
    shutil.rmtree(run_dir, ignore_errors=True)   # api_error 或 crash 重跑时清掉残留日志
    run_dir.mkdir(parents=True)
    slot = SLOTS[port] = {'sample_id': sid, 'seed': seed, 'log': run_dir / 'requests.jsonl',
                          'failed': 0, 'retried': 0, 'proxy_error': ''}
    cfg = EvalConfig(  # 与 dev-eval notebook 参数相同，只把时间预算换成教师的值
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=run_dir, submission_dir=AGENT_DIR,
        models=REGISTRIES[port], sandbox='subprocess', timeout_seconds=TIMEOUT_SECONDS,
        verification_timeout_seconds=VERIFY_TIMEOUT_SECONDS, max_time_minutes=TEACHER_MAX_TIME_MINUTES,
        max_tool_calls=MAX_TOOL_CALLS, max_turns=MAX_TURNS, limits=limits, generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                                                        token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', display_mode='quiet', verbose=False,
    )
    t0, result, crash = time.time(), None, ''
    try:
        result = await Evaluator(cfg).evaluate_task(task=TASKS[iid], task_index=k + 1, total_tasks=MAX_ATTEMPTS_PER_TASK)
    except Exception as e:   # 单题异常不能中断整个批次
        crash = f'{type(e).__name__}: {e}'
    finally:
        SLOTS.pop(port, None)

    patch = str(_maybe(result, 'agent_patch', 'patch') or '').strip()
    patch = '' if patch == 'NO_PATCH' else patch
    test_log = str(_maybe(result, 'test_output', 'test_log', 'verification_output', 'pytest_output') or '')
    error = str(_maybe(result, 'error', 'agent_error') or crash)
    (run_dir / 'agent.patch').write_text(patch, encoding='utf-8')
    (run_dir / 'test_output.log').write_text(test_log, encoding='utf-8')
    reqs = read_jsonl(slot['log'])
    resolved = bool(_maybe(result, 'resolved'))
    rec = {'sample_id': sid, 'instance_id': iid, 'repo': RAW[iid]['repo'], 'seed': seed, 'model': MODEL_NAME,
           'usage': summarize_usage(reqs), 'wall_seconds': round(time.time() - t0, 1),
           'harness_tool_calls': _maybe(result, 'tool_calls'), 'resolved': resolved, 'error': error[:500]}

    def done(status: str, reason: str = '', **extra) -> dict:
        return {**rec, 'status': status, 'reason': reason, **extra}

    if crash:
        return done('crash', crash[:300])
    if not reqs:
        return done('crash', 'no_llm_requests: harness did not reach the logging proxy')
    if slot['proxy_error']:   # 代理代码本身的问题，不是 API 故障；会重跑，但应先修代码
        return done('crash', f"proxy_error: {slot['proxy_error']}")
    if slot['failed'] or API_ERR_RE.search(error):
        return done('api_error', f"{slot['failed']} request(s) failed after retries; harness: {error[:150]}")
    messages, tools, usage, why = build_trajectory(reqs)
    if why == 'no_submit' and slot['retried']:   # 没提交且中途遇到过限流，可能是 API 拖慢导致超时，下次重跑
        return done('api_error', 'retried_and_no_submit')
    if why:
        return done('rejected', why)
    ver = verify(iid, patch, resolved, test_log, error)
    if not ver['passed']:
        return done('rejected', f"unverified_{ver['mode']}", verify=ver)
    return done('success', verify=ver, quality=quality(messages, usage), messages=messages, tools=tools, patch=patch)


start_proxies()
REGISTRIES = {PORT_BASE + w: make_registry(PORT_BASE + w) for w in range(MAX_WORKERS)}
print(f'{MAX_WORKERS} logging proxies on ports {PORT_BASE}..{PORT_BASE + MAX_WORKERS - 1} -> {BASE_URL} ({MODEL_NAME})')

## 5. Batch Execution & Summary
`MAX_WORKERS` 道题并发；同一道题按 seed 串行，攒够 `TARGET_PER_TASK` 条合格轨迹后停止。每个 attempt 结束后立即追加写入结果。最后打印汇总并打包 `d1_results.zip`。

In [ ]:
# ==== 5. Batch Execution & Summary ====
from tqdm.auto import tqdm

RUN_COST = {'usd': 0.0}                                                        # 本次运行的累计费用
WORST_ATTEMPT_H = (TEACHER_MAX_TIME_MINUTES + VERIFY_TIMEOUT_SECONDS / 60 + 3) / 60   # 智能体 + 验证 + 沙箱准备


def fuse_reason() -> str:
    if RUN_COST['usd'] >= MAX_COST_USD:
        return f'cost {RUN_COST["usd"]:.2f} >= MAX_COST_USD {MAX_COST_USD}'
    if (time.time() - NOTEBOOK_START) / 3600 + WORST_ATTEMPT_H > MAX_WALL_HOURS:
        return f'wall clock would exceed MAX_WALL_HOURS {MAX_WALL_HOURS}'
    return ''


async def worker(w: int, queue: asyncio.Queue, pbar) -> None:
    """依次领取题目；同一题的 attempt 按 seed 串行执行，攒够 TARGET_PER_TASK 条合格轨迹即停。"""
    port = PORT_BASE + w
    while not queue.empty():
        iid = queue.get_nowait()
        for k in pending_attempts(iid):
            if n_qualified(iid) >= TARGET_PER_TASK or fuse_reason():
                break
            rec = await run_attempt(iid, k, port)
            if rec['status'] == 'success':        # 先写 SFT，再写台账（续跑时以 SFT 文件为准）
                append_jsonl(SFT_PATH, rec)
            row = {k_: v for k_, v in rec.items() if k_ not in ('messages', 'tools', 'patch')}
            append_jsonl(LEDGER_PATH, row)
            ledger[rec['sample_id']] = row
            RUN_COST['usd'] += rec['usage']['cost_usd']
            q = rec.get('quality', {})
            note = rec['reason'] or ('qualified' if q['qualified'] else 'unqualified:' + ','.join(q['failed_rules']))
            tqdm.write(f"[w{w}] {rec['sample_id']:<22} {rec['status']:<9} {note} | "
                       f"${rec['usage']['cost_usd']:.4f} {rec['wall_seconds']:.0f}s")
            pbar.update(1)
            pbar.set_postfix(cost=f"${RUN_COST['usd']:.2f}", qualified=sum(n_qualified(i) for i in D1_IDS))
        if fuse_reason():
            tqdm.write(f'[w{w}] stop: {fuse_reason()}')
            return


async def main() -> None:
    queue = asyncio.Queue()
    for iid in D1_IDS:
        if pending_attempts(iid):
            queue.put_nowait(iid)
    with tqdm(total=sum(len(pending_attempts(i)) for i in D1_IDS), desc='attempts (upper bound)') as pbar:
        await asyncio.gather(*(worker(w, queue, pbar) for w in range(MAX_WORKERS)))


# notebook 内核已经有一个运行中的事件循环，所以在单独的线程里执行 asyncio.run（与 dev-eval 的 run_sync 相同）
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
    ex.submit(lambda: asyncio.run(main())).result()

# ---- 汇总 ----
rows = [r for r in ledger.values() if r['instance_id'] in D1_IDS]
succ = [r for r in rows if r['status'] == 'success']
qual = [r for r in succ if r['quality']['qualified']]
print(f"\nThis run cost ${RUN_COST['usd']:.2f}; cumulative (kept attempts) ${sum(r['usage']['cost_usd'] for r in rows):.2f}, "
      f"tokens in/out {sum(r['usage']['prompt_tokens'] for r in rows):,}/{sum(r['usage']['completion_tokens'] for r in rows):,}")
print('Attempts by status:', dict(Counter(r['status'] for r in rows)))
print(f'Trajectories: success {len(succ)}, qualified {len(qual)} | tasks at target {sum(n_qualified(i) >= TARGET_PER_TASK for i in D1_IDS)}/{len(D1_IDS)}')
for repo in sorted({RAW[i]['repo'] for i in D1_IDS}):
    ids = [i for i in D1_IDS if RAW[i]['repo'] == repo]
    print(f"  {repo:<18} tasks solved {len({r['instance_id'] for r in succ if r['repo'] == repo})}/{len(ids)}, "
          f"with qualified {len({r['instance_id'] for r in qual if r['repo'] == repo})}/{len(ids)}, "
          f"qualified trajs {sum(r['repo'] == repo for r in qual)}")
print('Non-success reasons:', Counter(f"{r['status']}:{r['reason'].split(':')[0]}" for r in rows if r['status'] != 'success').most_common(10))
print('Failed quality rules (success only):', Counter(x for r in succ for x in r['quality']['failed_rules']).most_common())
print('Unfinished tasks:', [i for i in D1_IDS if pending_attempts(i) and n_qualified(i) < TARGET_PER_TASK][:20])

archive = shutil.make_archive(str(OUT_DIR.parent / 'd1_results'), 'zip', root_dir=OUT_DIR)
print(f'\nSFT file: {SFT_PATH} ({len(read_jsonl(SFT_PATH))} rows) | ledger: {LEDGER_PATH} | archive: {archive}')